the architecture of the visual cortex

they showed that many neurones in visual cortex react only to image,and then they can overlap,and showed also that there are neurones that react only to horizontal line and one's for vertical line

In [1]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images
sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

In [ ]:
sample_images.shape

torch.Size([2, 427, 640, 3])

In [3]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)

In [4]:
import torchvision
import torchvision.transforms.v2 as T
cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [5]:
import torch.nn as nn 
torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
fmaps = conv_layer(cropped_images)
fmaps.shape

torch.Size([2, 32, 64, 114])

we lose 6 pixels in height and width because we dont activation padding and for that we crop some pixels to apply convulution

In [6]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7,padding="same")
fmaps = conv_layer(cropped_images)
fmaps.shape

torch.Size([2, 32, 70, 120])

In [7]:
max_pool = nn.MaxPool2d(kernel_size=2)

In [8]:
import torch.nn.functional as F
class DepthPool(torch.nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding
    def forward(self, inputs):
        batch, channels, height, width = inputs.shape
        Z = inputs.view(batch, channels, height * width) # merge spatial dims
        Z = Z.permute(0, 2, 1) # switch spatial and channels dims
        Z = F.max_pool1d(Z, kernel_size=self.kernel_size,
        stride=self.stride,
        padding=self.padding) # compute max pool
        Z = Z.permute(0, 2, 1) # switch back spatial and channels dims
        return Z.view(batch, -1, height, width) # unmerge spatial dims

In [9]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)

In [10]:
output

tensor([[[[0.6434]],

         [[0.5972]],

         [[0.5825]]],


        [[[0.7631]],

         [[0.2601]],

         [[0.1085]]]])

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
from functools import partial
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")
model = nn.Sequential(
DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7),
nn.ReLU(),
nn.MaxPool2d(kernel_size=2),
DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
nn.MaxPool2d(kernel_size=2),
DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
nn.MaxPool2d(kernel_size=2),
nn.Flatten(),
nn.Linear(in_features=2304, out_features=128), nn.ReLU(),
nn.Dropout(0.5),
nn.Linear(in_features=128, out_features=64), nn.ReLU(),
nn.Dropout(0.5),
nn.Linear(in_features=64, out_features=10),
).to(device)